# TUGAS MANDIRI

## Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu prediksi omzet harian tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi total_penjualan sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_kopi = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

26/11/18 19:27:43 WARN Utils: Your hostname, rindani-Latitude-3410 resolves to a loopback address: 127.0.1.1; using 192.168.1.11 instead (on interface wlp0s20f3)
26/11/18 19:27:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 19:27:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### A. Feature Engineering 

Gunakan StringIndexer untuk mengubah kolom nama_menu, cabang, dan waktu_pesan menjadi indeks numerik. Gabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom hasil indexing) menjadi satu kolom features menggunakan VectorAssembler.

In [3]:
model_idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx").fit(df_kopi)
model_idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_kopi)
model_idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx").fit(df_kopi)

df_kopi_idx = model_idx_menu.transform(df_kopi)
df_kopi_idx = model_idx_cabang.transform(df_kopi_idx)
df_kopi_idx = model_idx_waktu.transform(df_kopi_idx)

assembler_kopi = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)
df_kopi_final = assembler_kopi.transform(df_kopi_idx)

### B. Train/Test Split & Pelatihan Model 

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

In [4]:
train_kopi, test_kopi = df_kopi_final.randomSplit([0.75, 0.25], seed=101)

lr_kopi = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_kopi_utama = lr_kopi.fit(train_kopi)

### C. Evaluasi Model 

Hitung dan tampilkan RMSE dan R² dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [5]:
pred_kopi_utama = model_kopi_utama.transform(test_kopi)

eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = eval_rmse.evaluate(pred_kopi_utama)
r2_utama = eval_r2.evaluate(pred_kopi_utama)

print("=== HASIL EVALUASI MODEL UTAMA (5 FITUR) ===")
print(f"RMSE : Rp {rmse_utama:,.0f}")
print(f"R²   : {r2_utama:.4f}")

=== HASIL EVALUASI MODEL UTAMA (5 FITUR) ===
RMSE : Rp 10,901
R²   : 0.9107


### D. Eksperimen Perbandingan Fitur 

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

### E. Prediksi Skenario Baru 

Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training — perhatikan: gunakan model StringIndexer yang sama yang sudah di-fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan untuk ketiga skenario tersebut.